# FLUX Image-to-Image Depth Batch 🗒️🖼️

⚠️ **Remember to copy this notebook in your Drive and rename.**

🤗 This notebook uses [Hugging Face Diffusers](https://huggingface.co/docs/diffusers/en/index) to create pipelines for tasks such as image generation.

*Workflows for IAAC MaCDA GenAI  (Apr - Jun 2026) taught by [James McBennett](https://www.linkedin.com/in/mcbennett/) and [Aymeric Brouez](https://www.linkedin.com/in/aymeric-brouez/)*

*With special thanks to past faculty [Nono Martínez Alonso](https://youtube.com/NonoMartinezAlonso).*

***Workflow requires A100 runtime**


### Mount Drive

In [ ]:
# Mount Drive
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

## Hugging Face Token

In [ ]:
# Sign up at Hugging Face and create a "Read" access token (not the default "Fine-Grained" token).
# Click the 🔑 "Secrets" icon in the left sidebar.
# Enable Notebook Access, Set the Name to "HF_TOKEN", Paste your token as the Value

from google.colab import userdata
hf_token = userdata.get("HF_TOKEN")

## Setup

In [ ]:
%cd /content
!rm -rf iaac_genai
!git clone https://github.com/jamesmcbennett/iaac_genai
%cd /content/iaac_genai/

In [ ]:
import sys
sys.path.append('/content/iaac_genai')

In [ ]:
!pip install -q -r requirements.txt --quiet > /dev/null 2>&1

In [ ]:
from config import Config
from utils import set_image_path, save_image, save_yml, save_svg, save_gif, export_montage
from IPython.display import Image as IPythonImage
import torch

from diffusers.utils import load_image
from transformers import pipeline
from PIL import Image
import numpy as np
import os

## Set Directories

In [ ]:
# Input directory is a folder directory that will input all images inside that directory
# dataset from https://cmp.felk.cvut.cz/~tylecr1/facade/
INPUT_DIR = '/content/drive/MyDrive/iaac_genai/inputs/crow_museum'
Config.OUTPUT_DIR = '/content/drive/MyDrive/iaac_genai/outputs'

## Load Image

In [ ]:
images = []
resolution = 1024

for filename in os.listdir(INPUT_DIR ):
  if filename.lower().endswith((".jpg", ".jpeg", ".png", ".webp", ".bmp", ".tiff")):
    image_path = os.path.join(INPUT_DIR , filename)
    image = load_image(image_path)

    #Square
    w, h = image.size
    square_dim = min(w, h)
    image = image.crop((0, 0, square_dim, square_dim))

    #Resize
    max_dim = resolution
    image = image.resize((max_dim, max_dim), Image.LANCZOS)

    images.append(image)
    display(image)
    print("\n")

## Depth

In [ ]:
depth_images = []
#depth_estimator = pipeline('depth-estimation', model="Intel/dpt-large") #smaller lower filesize alternative
depth_estimator = pipeline('depth-estimation', model="LiheYoung/depth-anything-large-hf")

for image in images:
  depth_image = depth_estimator(image)['depth']
  depth_image_np = np.array(depth_image)
  depth_image_np = depth_image_np[:, :, None]
  depth_image_np = np.concatenate([depth_image_np, depth_image_np, depth_image_np], axis=2)
  depth_image_np = Image.fromarray(depth_image_np)

  display(depth_image_np)
  depth_images.append(depth_image_np)
  print("\n")

## Load pipeline

Load a pipeline with Hugging Face Diffusers.

In [ ]:
from diffusers import FluxControlNetPipeline, FluxControlNetModel

base_model = 'black-forest-labs/FLUX.1-dev'
controlnet_model_union = 'Shakker-Labs/FLUX.1-dev-ControlNet-Union-Pro-2.0'
controlnet = FluxControlNetModel.from_pretrained(controlnet_model_union, torch_dtype=torch.bfloat16)
pipe = FluxControlNetPipeline.from_pretrained(base_model, controlnet=controlnet, torch_dtype=torch.bfloat16)
pipe.to("cuda")

## Config

You can override parameters here.

In [ ]:
Config.PROMPT = 'A photorealistic wooden lattice building in a city with natural lighting and intricate structure.'
Config.SEED = 7797676568
Config.STEPS = 28

Config.AUTHOR = 'James'
Config.ALGO_TYPE = 'Image to Image'
Config.ALGO_NAME = 'Flux.1 Dev Depth'

Config.check()

## Generate

In [ ]:
frames = []

for i, depth_image in enumerate(depth_images):
  # Generate
  controlnet_conditioning_scale=0.7
  control_guidance_end=0.8
  guidance_scale=3.5

  generator = torch.Generator(Config.TORCH_DEVICE).manual_seed(Config.SEED)
  image = pipe(Config.PROMPT, control_image=depth_image, height=resolution, width=resolution, controlnet_conditioning_scale=controlnet_conditioning_scale, control_guidance_end=control_guidance_end, num_inference_steps=Config.STEPS, guidance_scale=guidance_scale, max_sequence_length=256, generator=generator).images[0]
  set_image_path()

  display(image)

  # what to add to frames to be used in gif / montage below
  frames.append(images[i])
  frames.append(depth_image)
  frames.append(image)

  # Save yml metadata
  save_yml(pipe)

  # Save image
  save_image(image)

  # Save svg parameters image
  save_svg({
      'SEED': Config.SEED,
      'STEPS': Config.STEPS,
      'Google Colab': '',
  })

## Gif

In [ ]:
Config.FPS = 1
gif_path = save_gif(frames)
IPythonImage(filename=gif_path)

## Montage

In [ ]:
### Cell be removed when export_montage is updated on Github

import imageio
import os

def export_montage(frames, columns):

    # Calculate the total number of frames and the number of frames per row
    total_frames = len(frames)
    frames_per_row = min(total_frames, columns)

    # Calculate the width and height of each frame in the montage
    frame_width, frame_height = frames[0].size  # Assuming all frames have the same size
    montage_width = frame_width * frames_per_row
    montage_height = frame_height * ((total_frames - 1) // frames_per_row + 1)

    # Create a blank image to hold the montage
    montage = np.zeros((montage_height, montage_width, 3), dtype=np.uint8)

    # Fill the montage with frames
    for idx, frame in enumerate(frames):
        row_idx = idx // frames_per_row
        col_idx = idx % frames_per_row
        start_x = col_idx * frame_width
        start_y = row_idx * frame_height
        end_x = start_x + frame_width
        end_y = start_y + frame_height
        montage[start_y:end_y, start_x:end_x, :] = np.array(frame)

    # Save the montage as a single image
    set_image_path()
    montage_path = f"{Config.IMAGE_PATH.replace('.png', '_montage.png')}"
    imageio.imwrite(montage_path, montage)

    return montage_path

In [ ]:
columns = 3

montage_path = export_montage(frames, columns)
IPythonImage(filename=montage_path)

## Disconnect

In [ ]:
from google.colab import runtime
runtime.unassign()